# 🧩 Mini-Lab: Choosing the Right Interface

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** the three main interface forms for delivering an AI application: CLI, HTTP API, and both
2. **Compare** tradeoffs (audience, latency, composability) across interface types
3. **Choose** the right interface for a given project scenario and justify the decision

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Workflow-Form Choice | Explore in a notebook, automate once in a script, reuse reliably as a package with tests, or expose it as a service that other systems can call. |


## Why Interface Choice Matters

You've built the **core logic** — maybe a summariser, an extractor, or a RAG pipeline. Now you need to let someone (or something) **use** it. The interface you wrap around your core determines:

- **Who** can call it (a developer in a terminal? a frontend app? a cron job?)
- **How** results flow (stdout, JSON over HTTP, exit codes)
- **Where** it runs (laptop, CI, cloud server)

Picking the wrong form wastes effort. Picking the right one can be a one-afternoon task.

## The Three Interface Forms

| Form | Entry Point | Typical Consumer | Best For |
|------|-------------|------------------|----------|
| **CLI** | `python main.py --arg value` | Developer, script, cron, CI | Batch jobs, automation, local tools |
| **HTTP API** | `POST /api/summarise` | Frontend, mobile app, other services | Real-time requests, multi-client access |
| **Both** | CLI *and* HTTP share the same core | Mixed audiences | Projects that start local then go online |

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## A Decision Framework

We'll encode the tradeoffs into a simple scoring model. For each scenario you answer three questions, and the scores point you toward CLI, HTTP, or both.

In [2]:
# Decision dimensions and how each interface scores on them.
# Scores are illustrative (1 = weak fit, 3 = strong fit).

DIMENSIONS = {
    "human_at_terminal": {
        "question": "Is the primary user a developer or ops person at a terminal?",
        "scores": {"CLI": 3, "HTTP": 1, "Both": 2},
    },
    "called_by_other_services": {
        "question": "Will other services (frontend, mobile, microservices) call it over the network?",
        "scores": {"CLI": 1, "HTTP": 3, "Both": 3},
    },
    "batch_or_pipeline": {
        "question": "Does it run as a batch/pipeline step (cron, CI, scripts)?",
        "scores": {"CLI": 3, "HTTP": 1, "Both": 2},
    },
    "multiple_clients_concurrently": {
        "question": "Do multiple clients need to hit it at the same time?",
        "scores": {"CLI": 1, "HTTP": 3, "Both": 3},
    },
    "quick_local_iteration": {
        "question": "Is fast local iteration (edit → run → check) a priority?",
        "scores": {"CLI": 3, "HTTP": 2, "Both": 2},
    },
}

In [3]:
def recommend_interface(answers: dict[str, bool]) -> str:
    """Given yes/no answers for each dimension, return a recommendation.

    Args:
        answers: Maps dimension key -> True (yes) or False (no).

    Returns:
        Markdown-formatted recommendation string.
    """
    totals = {"CLI": 0, "HTTP": 0, "Both": 0}

    for dim_key, is_yes in answers.items():
        if not is_yes:
            continue  # Only accumulate score for "yes" answers
        scores = DIMENSIONS[dim_key]["scores"]
        for form, score in scores.items():
            totals[form] += score

    ranked = sorted(totals.items(), key=lambda x: x[1], reverse=True)
    winner = ranked[0][0]

    lines = ["### Scores\n"]
    lines.append("| Interface | Score |")
    lines.append("|-----------|-------|")
    for form, score in ranked:
        marker = " ⬅️ recommended" if form == winner else ""
        lines.append(f"| **{form}** | {score}{marker} |")

    lines.append(f"\n**Recommendation → {winner}**")
    return "\n".join(lines)

## Scenario 1 — Nightly Report Summariser

> *A script runs every night via cron, reads new documents from a folder, summarises them, and writes a report file. No web users.*

In [4]:
scenario_1 = {
    "human_at_terminal":           True,   # Dev runs & debugs it locally
    "called_by_other_services":     False,  # No network consumers
    "batch_or_pipeline":            True,   # Runs via cron every night
    "multiple_clients_concurrently": False, # Single scheduled job
    "quick_local_iteration":        True,   # Developer iterates on prompts
}

md(recommend_interface(scenario_1))

### Scores

| Interface | Score |
|-----------|-------|
| **CLI** | 9 ⬅️ recommended |
| **Both** | 6 |
| **HTTP** | 4 |

**Recommendation → CLI**

CLI wins here: the consumer is a cron job and a developer. No web traffic, no concurrent clients.

## Scenario 2 — Customer Support Chat Widget

> *A React frontend sends user messages to a backend that calls an LLM and streams answers back. Hundreds of users may be active simultaneously.*

In [5]:
scenario_2 = {
    "human_at_terminal":           False,  # End users are on a web page
    "called_by_other_services":     True,   # React frontend calls API
    "batch_or_pipeline":            False,  # Real-time, not batch
    "multiple_clients_concurrently": True,  # Hundreds of users at once
    "quick_local_iteration":        False,  # Iteration matters less than uptime
}

md(recommend_interface(scenario_2))

### Scores

| Interface | Score |
|-----------|-------|
| **HTTP** | 6 ⬅️ recommended |
| **Both** | 6 |
| **CLI** | 2 |

**Recommendation → HTTP**

HTTP wins: the consumer is a web frontend, concurrency is a must, and there's no terminal user.

## Scenario 3 — Internal RAG Tool

> *An engineering team queries a knowledge base. Some use a terminal; others use an internal web dashboard. A CI job also tests retrieval quality nightly.*

In [6]:
scenario_3 = {
    "human_at_terminal":           True,   # Engineers at terminal
    "called_by_other_services":     True,   # Web dashboard calls API
    "batch_or_pipeline":            True,   # CI quality checks
    "multiple_clients_concurrently": True,  # Dashboard has several users
    "quick_local_iteration":        True,   # Engineers iterate on queries
}

md(recommend_interface(scenario_3))

### Scores

| Interface | Score |
|-----------|-------|
| **Both** | 12 ⬅️ recommended |
| **CLI** | 11 |
| **HTTP** | 10 |

**Recommendation → Both**

"Both" wins: the same core logic needs to serve a terminal user, a web dashboard, **and** a CI pipeline. The key architecture insight is:

```
┌─────────┐     ┌────────────┐
│  CLI    │────▶│            │
└─────────┘     │  Core      │
┌─────────┐     │  Logic     │
│  HTTP   │────▶│ (shared)   │
└─────────┘     └────────────┘
```

Both the CLI and the HTTP endpoint import and call the **same core function**. The interface is just a thin wrapper — it parses input, calls the core, and formats the output for its medium.

## The Key Principle: Separate Core from Interface

No matter which form you choose, the decision is easy if your **core logic is a plain function** that knows nothing about `argparse` or `FastAPI`.

```python
# core.py  — no CLI or HTTP imports here
def summarise(text: str, max_sentences: int = 3) -> str:
    """Pure business logic."""
    ...
```

```python
# cli.py  — thin CLI wrapper
from core import summarise
import argparse, sys

parser = argparse.ArgumentParser()
parser.add_argument("text")
args = parser.parse_args()
print(summarise(args.text))
```

```python
# api.py  — thin HTTP wrapper
from core import summarise
from fastapi import FastAPI

app = FastAPI()

@app.post("/summarise")
def endpoint(text: str):
    return {"summary": summarise(text)}
```

This separation means your choice of interface is **reversible** — you can always add the other form later with minimal effort.

## Quick-Reference Cheat Sheet

In [7]:
cheat_sheet = """
| Signal | Start with |
|--------|------------|
| Only developers/scripts will use it | **CLI** |
| A web or mobile frontend needs it | **HTTP** |
| It runs in CI / cron / pipelines | **CLI** |
| Multiple concurrent users | **HTTP** |
| Both terminal users AND web consumers | **Both** (shared core) |
| You're unsure and it's early | **CLI first** — easiest to add HTTP later |
"""

md(cheat_sheet)


| Signal | Start with |
|--------|------------|
| Only developers/scripts will use it | **CLI** |
| A web or mobile frontend needs it | **HTTP** |
| It runs in CI / cron / pipelines | **CLI** |
| Multiple concurrent users | **HTTP** |
| Both terminal users AND web consumers | **Both** (shared core) |
| You're unsure and it's early | **CLI first** — easiest to add HTTP later |


## 🎯 Summary

### Key Takeaways

1. **Workflow-form choice** — deciding between CLI, HTTP, or both is one of the first delivery decisions and it shapes how your application is consumed
2. **Match the form to the consumer** — terminal users and scripts favor CLI; web/mobile clients need HTTP; mixed audiences need both
3. **Separate core from interface** — keeping business logic in a plain function makes the interface choice reversible and lets you support multiple forms with thin wrappers
4. **When in doubt, start with CLI** — it's the simplest to build, test, and iterate on; you can always add an HTTP layer later